# Домашнее задание: Логистическая регрессия для классификации оттока клиентов

Цель задания: самостоятельно пройти все этапы решения задачи бинарной классификации с помощью логистической регрессии.

Датасет: Bank Customer Churn (отток клиентов банка).

Задача: предсказать, уйдет ли клиент из банка (закроет счет) или останется.

Что нужно сделать:
1. Загрузить и изучить данные.
2. Провести предобработку (очистка, кодирование категориальных признаков).
3. Разделить данные на обучающую и тестовую выборки.
4. Обучить модель логистической регрессии.
5. Оценить качество модели по метрикам accuracy, precision, recall, F1-score.
6. Интерпретировать результаты и сделать выводы.


# Шаг 1. Импорт библиотек

Задание: импортируйте все необходимые библиотеки для работы с данными, обучения модели и оценки качества.

Что нужно импортировать:
- pandas для работы с таблицами
- numpy для числовых операций
- train_test_split для разделения данных
- LogisticRegression для модели
- accuracy_score, precision_score, recall_score, f1_score, confusion_matrix для метрик


In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, confusion_matrix)


# Шаг 2. Загрузка данных

Датасет: Bank Customer Churn

Ссылка на данные:
https://raw.githubusercontent.com/YBI-Foundation/Dataset/main/Bank%20Churn%20Modelling.csv

Задание:
1. Загрузите датасет с помощью pd.read_csv.
2. Выведите первые 5 строк с помощью метода head().
3. Проверьте размер датасета с помощью len() или shape.

Описание столбцов:
- Attrition_Flag: целевая переменная (Existing Customer остался, Attrited Customer ушел)
- Customer_Age: возраст клиента
- Gender: пол клиента
- Dependent_count: количество иждивенцев
- Education_Level: уровень образования
- Marital_Status: семейное положение
- Income_Category: категория дохода
- Card_Category: тип карты
- Months_on_book: сколько месяцев клиент с банком
- Total_Relationship_Count: количество продуктов банка, которыми пользуется клиент
- Months_Inactive_12_mon: количество неактивных месяцев за последний год
- Contacts_Count_12_mon: сколько раз клиент обращался в банк за последний год
- Credit_Limit: кредитный лимит
- Total_Revolving_Bal: общий возобновляемый баланс
- Avg_Open_To_Buy: средняя доступная сумма для покупок
- Total_Amt_Chng_Q4_Q1: изменение суммы транзакций между 4 и 1 кварталом
- Total_Trans_Amt: общая сумма транзакций за последние 12 месяцев
- Total_Trans_Ct: общее количество транзакций за последние 12 месяцев
- Total_Ct_Chng_Q4_Q1: изменение количества транзакций между 4 и 1 кварталом
- Avg_Utilization_Ratio: средний коэффициент использования кредитной линии


In [2]:
url = "https://raw.githubusercontent.com/YBI-Foundation/Dataset/main/Bank%20Churn%20Modelling.csv"
df = pd.read_csv(url)
df.head()

df.shape
len(df)

10000

# Шаг 3. Первичный анализ данных

Задание:
1. Выполните метод info() для просмотра типов данных и пропусков.
2. Выполните describe(include='all') для базовой статистики.
3. Проверьте распределение целевой переменной Attrition_Flag с помощью value_counts().

Вопросы для анализа:
- Сколько всего клиентов в датасете? 10000
- Есть ли пропущенные значения? Нет
- Какая доля клиентов ушла из банка? 20%
- Сбалансированы ли классы? Нет, классы не сбалансированы, 80:20


In [7]:
df.info()
df.describe(include='all')
df['Churn'].value_counts()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   CustomerId        10000 non-null  int64  
 1   Surname           10000 non-null  object 
 2   CreditScore       10000 non-null  int64  
 3   Geography         10000 non-null  object 
 4   Gender            10000 non-null  object 
 5   Age               10000 non-null  int64  
 6   Tenure            10000 non-null  int64  
 7   Balance           10000 non-null  float64
 8   Num Of Products   10000 non-null  int64  
 9   Has Credit Card   10000 non-null  int64  
 10  Is Active Member  10000 non-null  int64  
 11  Estimated Salary  10000 non-null  float64
 12  Churn             10000 non-null  int64  
dtypes: float64(2), int64(8), object(3)
memory usage: 1015.8+ KB


,count
Churn,
0,7963
1,2037


# Шаг 4. Предобработка данных

Задание:
1. Удалите ненужные столбцы:
   - CLIENTNUM (идентификатор клиента, не несет информации для модели)
   - Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_1
   - Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_2
   (это технические столбцы, они не нужны)

2. Преобразуйте целевую переменную Attrition_Flag:
   - Existing Customer -> 0 (клиент остался)
   - Attrited Customer -> 1 (клиент ушел)

3. Определите, какие столбцы являются категориальными (тип object), а какие числовыми.

4. Примените one-hot encoding к категориальным признакам с помощью pd.get_dummies().
   Используйте параметр drop_first=True, чтобы избежать мультиколлинеарности.

Объяснение:
- One-hot encoding превращает категориальные признаки в набор бинарных столбцов.
- Например, столбец Gender с значениями M и F превратится в Gender_M (1 если мужчина, 0 если женщина).
- drop_first=True удаляет одну категорию из каждого признака, чтобы избежать избыточности.


In [8]:
df = df.drop(columns=['CustomerId', 'Surname'])



In [9]:
df['Churn'].value_counts()



,count
Churn,
0,7963
1,2037


In [13]:
categorical_cols = df.select_dtypes(include='object').columns
print(categorical_cols.tolist())

numerical_cols = df.select_dtypes(exclude='object').columns
print(numerical_cols.tolist())



['Geography', 'Gender']
['CreditScore', 'Age', 'Tenure', 'Balance', 'Num Of Products', 'Has Credit Card', 'Is Active Member', 'Estimated Salary', 'Churn']


In [12]:
df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)



# Шаг 5. Подготовка данных для обучения

Задание:
1. Создайте матрицу признаков X:
   - Удалите из датасета столбец с целевой переменной Attrition_Flag.
   - Преобразуйте результат в numpy массив с помощью .values.

2. Создайте вектор целевой переменной y:
   - Выделите столбец Attrition_Flag.
   - Преобразуйте в numpy массив.

3. Выведите размеры X и y с помощью .shape.

Объяснение:
- X содержит все признаки (независимые переменные), по которым модель будет делать предсказания.
- y содержит целевую переменную (зависимую переменную), которую модель будет предсказывать.
- Размер X должен быть (количество клиентов, количество признаков).
- Размер y должен быть (количество клиентов,).


In [14]:
X = df_encoded.drop(columns=['Churn']).values
y = df_encoded['Churn'].values



# Шаг 6. Разделение на обучающую и тестовую выборки

Задание:
1. Разделите данные на train и test с помощью train_test_split.
2. Используйте следующие параметры:
   - test_size=0.2 (20% данных для теста, 80% для обучения)
   - random_state=42 (для воспроизводимости результатов)
   - stratify=y (чтобы пропорции классов в train и test были одинаковыми)

3. Выведите размеры полученных массивов.

Объяснение:
- train данные используются для обучения модели.
- test данные используются для оценки качества модели на новых данных.
- stratify=y гарантирует, что в обеих выборках будет примерно одинаковое соотношение ушедших и оставшихся клиентов.
- random_state=42 делает разделение повторяемым (если запустить код снова, получим те же самые train и test).


In [16]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print("Размер X_train:", X_train.shape)
print("Размер X_test:", X_test.shape)
print("Размер y_train:", y_train.shape)
print("Размер y_test:", y_test.shape)


Размер X_train: (8000, 11)
Размер X_test: (2000, 11)
Размер y_train: (8000,)
Размер y_test: (2000,)


# Шаг 7. Обучение модели логистической регрессии

Задание:
1. Создайте объект модели LogisticRegression с параметром max_iter=1000.
2. Обучите модель на обучающих данных с помощью метода fit(X_train, y_train).
3. Выведите сообщение об успешном обучении.

Объяснение:
- LogisticRegression это алгоритм для бинарной классификации.
- max_iter=1000 задает максимальное число итераций для сходимости алгоритма оптимизации.
- Метод fit обучает модель на данных X_train (признаки) и y_train (целевая переменная).
- После обучения модель готова делать предсказания.


In [18]:
model = LogisticRegression(max_iter=1000)

model.fit(X_train, y_train)



/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression(max_iter=1000)

# Шаг 8. Предсказания на тестовой выборке

Задание:
1. Сделайте предсказания для тестовой выборки с помощью метода predict(X_test).
2. Получите вероятности предсказаний с помощью метода predict_proba(X_test).
3. Выведите первые 10 предсказаний и первые 10 реальных значений для сравнения.

Объяснение:
- predict(X_test) возвращает предсказанный класс для каждого клиента (0 или 1).
- predict_proba(X_test) возвращает вероятности для каждого класса. Первый столбец вероятность класса 0, второй вероятность класса 1.
- Сравнение предсказаний с реальными значениями помогает визуально оценить, насколько хорошо работает модель.


In [20]:
y_pred = model.predict(X_test)

print("Первые 10 предсказаний:", y_pred[:10])
print("Первые 10 реальных значений:", y_test[:10])

y_proba = model.predict_proba(X_test)
print("Первые 10 вероятностей:\n", y_proba[:10])


Первые 10 предсказаний: [0 0 0 0 0 0 0 0 0 0]
Первые 10 реальных значений: [0 0 0 0 0 0 0 0 0 0]
Первые 10 вероятностей:
 [[0.88412628 0.11587372]
 [0.68909518 0.31090482]
 [0.84641387 0.15358613]
 [0.83688968 0.16311032]
 [0.86596568 0.13403432]
 [0.81505591 0.18494409]
 [0.89916447 0.10083553]
 [0.62566585 0.37433415]
 [0.66167226 0.33832774]
 [0.79305617 0.20694383]]
